# Context Managers

## Implementation of a context manager with magic methods

Create a `StdoutRedirector` class that can be used as a context manager. This class will store the value of `sys.stdout` when entering the context, and replace it with the argument given to it at creation. When leaving the context, the old standard output must be restored. `__enter__` will return the new stream, so that `as` can get it.

With an in-memory stream, this code must print `Captured: 'Hello\n'`:

```python
with StdoutRedirector(io.StringIO()) as buffer:
  print("Hello")
print(f"Captured: {buffer.getvalue()!r}")
```

**Useful documentation**

- [`sys.stdout`](https://docs.python.org/3/library/sys.html#sys.stdout)
- [`io.StringIO`](https://docs.python.org/3/library/io.html#io.StringIO)
- [The context manager type](https://docs.python.org/3/library/stdtypes.html#typecontextmanager)

In [ ]:
# Your code here

### Additional hints

`__enter__` keeps `sys.stdout` in an attribute before replacing it: `__exit__` needs it to restore it. What `__enter__` returns is what `as` receives. `__exit__` takes three arguments, which describe the exception if there is one, and returns `False` to let it propagate.

```python
class StdoutRedirector:
  def __init__(self, new_stdout):
    self.new_stdout = new_stdout

  def __enter__(self):
    self.old_stdout = ...
    sys.stdout = ...
    return ...

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = ...
    return False
```

### Solution

In [ ]:
import io
import sys


class StdoutRedirector:
  def __init__(self, new_stdout):
    self.new_stdout = new_stdout

  def __enter__(self):
    self.old_stdout = sys.stdout
    sys.stdout = self.new_stdout
    return self.new_stdout

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = self.old_stdout
    return False


with StdoutRedirector(io.StringIO()) as buffer:
  print("Hello")
print(f"Captured: {buffer.getvalue()!r}")

## Implementation of a context manager with a generator

Recreate this manager with `contextlib.contextmanager`, as a generator function `redirect_output(new_output)` that yields the new stream. Write it without `try`/`finally` first.

Then raise an exception in the body of the `with`, catch it around the `with`, and check whether `sys.stdout` is restored, by comparing it with `is` to its value from before the block. Keep that value in a variable, and put it back in `sys.stdout` yourself before printing the result: otherwise, if the output isn't restored, nothing will show in the notebook anymore.

Without `try`/`finally`, you must see that `sys.stdout` isn't restored. Add it, and check that it is.

In [ ]:
# Your code here

### Additional hints

The code before `yield` runs when entering the block. Entering stays outside the `try`: if it fails, there is nothing to restore. Only the `yield` goes in the `try`, and restoring in the `finally`.

```python
@contextlib.contextmanager
def redirect_output(new_output):
  old_output = ...
  sys.stdout = ...
  try:
    yield ...
  finally:
    ...


standard_output = sys.stdout
try:
  with redirect_output(io.StringIO()):
    raise ValueError("error in the body")
except ValueError:
  pass
restored = ...
sys.stdout = standard_output
print(f"Output restored: {restored}")
```

### Solution

In [ ]:
import contextlib
import io
import sys


@contextlib.contextmanager
def redirect_output_no_finally(new_output):
  old_output = sys.stdout
  sys.stdout = new_output
  yield new_output
  sys.stdout = old_output


@contextlib.contextmanager
def redirect_output(new_output):
  old_output = sys.stdout
  sys.stdout = new_output
  try:
    yield new_output
  finally:
    sys.stdout = old_output


def check_restoration(manager):
  standard_output = sys.stdout
  try:
    with manager(io.StringIO()):
      raise ValueError("error in the body")
  except ValueError:
    pass
  restored = sys.stdout is standard_output
  sys.stdout = standard_output
  print(f"{manager.__name__}: output restored: {restored}")


check_restoration(redirect_output_no_finally)
check_restoration(redirect_output)

In practice, `contextlib.redirect_stdout` already does all of this.

## Implementation of a Context That Suppresses a Type of Exception

Use `contextlib.contextmanager` to implement a context manager `suppress(*exception_types)` that suppresses the exceptions of the given types (stops their propagation) in the context's body, like `contextlib.suppress`. An exception of another type must go through the context:

```python
with suppress(NameError):
  print(abcd)  # Nothing is printed, and the program goes on

with suppress(NameError, KeyError):
  {}["missing"]  # Same

with suppress(NameError):
  1 / 0  # The ZeroDivisionError propagates
```

In [ ]:
# Your code here

### Additional hints

An exception raised in the body of the `with` is raised again in the generator, at the `yield`: that's where you catch it. `*exception_types` gives a tuple, which `except` accepts as is. An exception caught and not raised again doesn't propagate anymore.

```python
@contextlib.contextmanager
def suppress(*exception_types):
  try:
    ...
  except ...:
    ...
```

### Solution

In [ ]:
import contextlib


@contextlib.contextmanager
def suppress(*exception_types):
  try:
    yield
  except exception_types:
    pass


with suppress(NameError):
  print(abcd)

with suppress(NameError, KeyError):
  {}["missing"]

try:
  with suppress(NameError):
    1 / 0
except ZeroDivisionError:
  print("ZeroDivisionError isn't suppressed: it goes through the context")

## Rolling back a transaction

Write a `Transaction` class that protects a dictionary of accounts: it saves its content when entering the context, and returns the dictionary. If the body raises an exception, `__exit__` puts the dictionary back in its entry state and lets the exception propagate. Otherwise, the changes stay.

The `transfer` function below uses it. This code must leave the accounts intact on the second transfer, which fails once Bob has already been debited:

```python
accounts = {"alice": 100, "bob": 20}
transfer(accounts, "alice", "bob", 30)
print(accounts)  # {'alice': 70, 'bob': 50}
try:
  transfer(accounts, "bob", "alice", 80)
except ValueError as error:
  print(error)  # insufficient balance on bob
print(accounts)  # {'alice': 70, 'bob': 50}
```

**Useful documentation**

- [`dict.clear`](https://docs.python.org/3/library/stdtypes.html#dict.clear)
- [`dict.update`](https://docs.python.org/3/library/stdtypes.html#dict.update)

In [ ]:
def transfer(accounts, source, target, amount):
  with Transaction(accounts) as balances:
    balances[source] -= amount
    if balances[source] < 0:
      raise ValueError(f"insufficient balance on {source}")
    balances[target] += amount


# Your code here

### Additional hints

`__exit__` knows the body raised when `exc_type` isn't `None`. Restoring happens in place, with `clear` and `update`, because `accounts` stays the same dictionary for the caller: `self.accounts = self.backup` would only change the attribute. Returning `False` lets the exception propagate, without raising it again.

```python
class Transaction:
  def __init__(self, accounts):
    self.accounts = accounts

  def __enter__(self):
    self.backup = ...
    return ...

  def __exit__(self, exc_type, exc_val, exc_tb):
    if ...:
      self.accounts.clear()
      self.accounts.update(...)
    return ...
```

### Solution

In [ ]:
class Transaction:
  def __init__(self, accounts):
    self.accounts = accounts

  def __enter__(self):
    self.backup = dict(self.accounts)
    return self.accounts

  def __exit__(self, exc_type, exc_val, exc_tb):
    if exc_type is not None:
      self.accounts.clear()
      self.accounts.update(self.backup)
    return False


accounts = {"alice": 100, "bob": 20}
transfer(accounts, "alice", "bob", 30)
print(accounts)
try:
  transfer(accounts, "bob", "alice", 80)
except ValueError as error:
  print(error)
print(accounts)

## Opening a list of files (optional)

Write a function `first_lines(paths)` that opens all the files of the list at the same time, with `contextlib.ExitStack`, and returns the list of their first lines, without the line break. Also register with `callback` the printing of `Files closed`, which must happen even if an opening fails:

```python
print(first_lines(paths))  # Files closed, then ['one', 'two', 'three']
try:
  first_lines([*paths, "missing.txt"])
except FileNotFoundError as error:
  print(error)  # Files closed, then the error
```

In [ ]:
from pathlib import Path

paths = ["january.txt", "february.txt", "march.txt"]
for path, content in zip(paths, ["one", "two", "three"]):
  Path(path).write_text(content + "\n", encoding="utf-8")

# Your code here

### Additional hints

`stack.enter_context(manager)` enters the context and returns what its `__enter__` returns, here the open file. The stack leaves its contexts on its own exit, in reverse order: to print after the files are closed, the `callback` is registered before them.

```python
def first_lines(paths):
  with contextlib.ExitStack() as stack:
    stack.callback(...)
    files = [... for path in paths]
    return [... for file in files]
```

### Solution

In [ ]:
import contextlib


def first_lines(paths):
  with contextlib.ExitStack() as stack:
    stack.callback(print, "Files closed")
    files = [stack.enter_context(open(path, encoding="utf-8")) for path in paths]
    return [file.readline().strip() for file in files]


print(first_lines(paths))
try:
  first_lines([*paths, "missing.txt"])
except FileNotFoundError as error:
  print(error)

## Reimplementation of the `contextlib.contextmanager` decorator (advanced)

Using a decorator, which will create a class with the `__init__`, `__enter__` and `__exit__` methods, reimplement the `contextlib.contextmanager` decorator. Check it by decorating `suppress` with your `contextmanager`: the examples of its exercise must give the same result.

**Useful documentation**

- [`next`](https://docs.python.org/3/library/functions.html#next)
- [`generator.throw`](https://docs.python.org/3/library/stdtypes.html#generator.throw)
- [`StopIteration`](https://docs.python.org/3/library/exceptions.html#StopIteration)

In [ ]:
# Your code here

### Additional hints

The decorator returns a class: the call `suppress(NameError)` therefore creates an instance, whose `__init__` receives the arguments and creates the generator.

- `__enter__` calls `next` on the generator, which runs up to the `yield`: the yielded value is what `as` receives.
- Without an exception, `__exit__` calls `next` a second time: the generator resumes after the `yield` and must finish, that is raise `StopIteration`.
- With an exception, `__exit__` raises it again in the generator, at the `yield`, with `throw(exc_val)`. If the generator catches it and finishes, `StopIteration` is raised, and `__exit__` returns `True` to suppress it. If it doesn't catch it, it comes out of `throw` and propagates.
- If the generator yields a second time, no `StopIteration` is raised: that's an error, to report with a `RuntimeError`.

```python
def contextmanager(generator):
  class Wrapper:
    def __init__(self, *args, **kwargs):
      self._generator = ...

    def __enter__(self):
      return ...

    def __exit__(self, exc_type, exc_val, exc_tb):
      try:
        if exc_type is None:
          ...
        else:
          ...
      except StopIteration:
        pass
      else:
        raise RuntimeError(...)
      return True

  return Wrapper
```

### Solution

In [ ]:
def contextmanager(generator):
  class Wrapper:
    def __init__(self, *args, **kwargs):
      self._generator = generator(*args, **kwargs)

    def __enter__(self):
      return next(self._generator)

    def __exit__(self, exc_type, exc_val, exc_tb):
      try:
        if exc_type is None:
          next(self._generator)
        else:
          self._generator.throw(exc_val)
      except StopIteration:
        pass
      else:
        raise RuntimeError("Generator used as a context should yield only once")
      return True

  return Wrapper


@contextmanager
def suppress(*exception_types):
  try:
    yield
  except exception_types:
    pass


with suppress(NameError):
  print(abcd)

try:
  with suppress(NameError):
    1 / 0
except ZeroDivisionError:
  print("ZeroDivisionError isn't suppressed: it goes through the context")

The same mechanism as a video: who runs the code after the `yield`, and why it needs a `try`/`finally` around it: [`@contextmanager`, step by step](https://shuuchuu.github.io/videos/python/en/contextmanager-implementation.mp4).